# modeling_v13 — M4 최종 맞대결: **Conservative-GA(99) vs Balanced-GA(108)**

**목적**: M3에서 GA가 뽑은 두 후보를 **한 노트북(=한 환경)** 에서 재평가해
GroupKFold(C20) **절대 비교**를 성립시키고 프리셋을 확정한다.
(M3는 GA=Colab / RFECV=로컬이라 GKF 절대값이 환경차로 비교 불가였음.)

**후보 복원**: `fixed(core10+champion)` + `select_result_<preset>_GA.json` 의 `selected_features`.
**평가(고정)**: `M8_PARAMS`·705 로 KFold OOF(`fold_kf5`) + GroupKFold(C20). 둘 다 **같은 df/환경** → 직접 비교 가능.

> ⚠️ 전제: `v13_select_common.py`, `../modeling_v8/v8_timeline_common.py`, `../문제1(하)/train_data.csv`,
> `../pm_log.json`, `feature_diet_selected.json`, `data/v13_fdc_pool_wf_oof.csv.gz`,
> `select_result_Conservative_GA.json`, `select_result_Balanced_GA.json`.
> 런타임: 후보 2개 × (KFold5 + GKF5) = **20 LGBM fit**(705라운드). 로컬 ~30분 · **CPU 전용**.
> ※ 이 노트북 하나만 돌리면 두 후보가 자동으로 같은 환경 → GKF 공정 비교.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import json, time
import pandas as pd
import v13_select_common as sc

CANDS = ["Conservative", "Balanced"]

# 후보별: df/그룹 로드 + fixed 재구성 + JSON 선택피처로 최종 피처셋 복원
data = {}
for p in CANDS:
    df, y, groups, g = sc.load(p)                     # df/y/groups 는 프리셋 무관 동일
    selp = json.loads(open(f"select_result_{p}_GA.json", encoding="utf-8").read())["selected_features"]
    feats = list(dict.fromkeys(g["fixed"] + selp))    # core10+champion + GA선택
    ok, have = sc.floor_ok(feats)
    data[p] = dict(df=df, y=y, groups=groups, feats=feats, floor_ok=ok, have=have,
                   n_fixed=len(g["fixed"]), n_sel=len(selp))
    print(f"{p:12s} n_total={len(feats):3d} (fixed {len(g['fixed'])} + GA {len(selp)})  floor={ok} {have}")

Conservative n_total= 99 (fixed 15 + GA 84)  floor=True {'C17': 4, 'C11': 5, 'C31': 3, 'C15': 3, 'C16': 2}
Balanced     n_total=108 (fixed 15 + GA 93)  floor=True {'C17': 2, 'C11': 9, 'C31': 2, 'C15': 3, 'C16': 3}


## 재평가 — 두 후보, 같은 환경, M8_PARAMS·705 (KFold + GroupKFold)

In [2]:
rows = []
for p in CANDS:
    d = data[p]; t = time.time()
    kf = sc._oof_kfold(d["df"], d["feats"], d["y"], sc.tl.M8_PARAMS, sc.tl.BEST_ROUNDS)
    gk, _ = sc._oof_group(d["df"], d["feats"], d["y"], d["groups"], sc.tl.M8_PARAMS, sc.tl.BEST_ROUNDS, 5)
    rows.append([f"{p}-GA", d["n_total"] if False else len(d["feats"]),
                 round(kf, 3), round(gk, 3), d["floor_ok"], round(time.time()-t, 0)])
    print(f"{p:12s} KFold={kf:.3f}  GroupKFold(C20)={gk:.3f}  ({time.time()-t:.0f}s)")

res = pd.DataFrame(rows, columns=["candidate","n_total","KFold_OOF","GroupKFold_C20","floor_ok","sec"])
res

Conservative KFold=50.677  GroupKFold(C20)=71.366  (233s)
Balanced     KFold=51.412  GroupKFold(C20)=71.272  (235s)


,candidate,n_total,KFold_OOF,GroupKFold_C20,floor_ok,sec
0,Conservative-GA,99,50.677,71.366,True,233.0
1,Balanced-GA,108,51.412,71.272,True,235.0


## 판정 — GroupKFold(C20) 최소가 승자 (이제 같은 환경이라 절대 비교 성립)

In [3]:
res_sorted = res.sort_values("GroupKFold_C20").reset_index(drop=True)
win = res_sorted.iloc[0]
print(f"▶ 승자(정직-CV 기준): {win['candidate']}  GroupKFold={win['GroupKFold_C20']}  KFold={win['KFold_OOF']}  n={win['n_total']}")
print(f"  차이(GKF): {res_sorted.iloc[1]['GroupKFold_C20'] - win['GroupKFold_C20']:+.3f}")
print(f"  차이(KFold): Conservative vs Balanced = {res.set_index('candidate').loc['Conservative-GA','KFold_OOF'] - res.set_index('candidate').loc['Balanced-GA','KFold_OOF']:+.3f}")
res_sorted

▶ 승자(정직-CV 기준): Balanced-GA  GroupKFold=71.272  KFold=51.412  n=108
  차이(GKF): +0.094
  차이(KFold): Conservative vs Balanced = -0.735


,candidate,n_total,KFold_OOF,GroupKFold_C20,floor_ok,sec
0,Balanced-GA,108,51.412,71.272,True,235.0
1,Conservative-GA,99,50.677,71.366,True,233.0


## 저장

In [4]:
out = {"candidates": {r["candidate"]: {"n_total": int(r["n_total"]),
                                        "KFold_OOF": r["KFold_OOF"],
                                        "GroupKFold_C20": r["GroupKFold_C20"],
                                        "floor_ok": bool(r["floor_ok"])} for _, r in res.iterrows()},
       "winner_by_groupkfold": res.sort_values("GroupKFold_C20").iloc[0]["candidate"]}
json.dump(out, open("final_compare_results.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
res.to_csv("final_compare_results.csv", index=False)
print("saved: final_compare_results.json, final_compare_results.csv")

saved: final_compare_results.json, final_compare_results.csv


---
### 다음 단계
프리셋 확정 후 **그 셋을 고정하고 파라미터 재튜닝**(Optuna, GroupKFold 목적)으로 절대 성능 확정.
고정 M8_PARAMS(10피처용) 과적합을 풀면 KFold·GKF 둘 다 더 내려갈 여지.